# Detección y localización de grietas con aprendizaje débilmente supervisado

**Idea:** entrenamos un clasificador con etiquetas *de imagen completa* (grieta / sin grieta) y obtenemos la
**ubicación** de la grieta con **Grad-CAM**, sin haber usado nunca máscaras ni cajas durante el entrenamiento.
Eso es lo que hace al método *débilmente supervisado*.

**Cambios respecto a la versión anterior**

| # | Problema anterior | Corrección |
|---|---|---|
| 1 | Grad-CAM calculado sobre `modelo_base` (sin cabeza de clasificación) | Grad-CAM calculado sobre el logit de la cabeza `clasificador` |
| 2 | Faltaban `import numpy / cv2 / matplotlib` | Todos los imports en una sola celda |
| 3 | Clases desbalanceadas (15 % grietas) y solo se medía accuracy | Pesos por muestra + métricas AUC, precisión, recall, F1, matriz de confusión |
| 4 | Split aleatorio (recortes de la misma foto en train y val → fuga de datos) | Split **por foto de origen** en train / val / test |
| 5 | Base congelada, modelo subajustado | Fase 1 (cabeza) + Fase 2 (fine-tuning de los últimos bloques) |
| 6 | Mapa de calor de 8×8 (muy grueso para grietas finas) | Grad-CAM en 3 resoluciones (8×8, 16×16, 32×32) + fusión + binarización + cajas |
| 7 | Preprocesamiento `/255` | `preprocess_input` de MobileNetV2 (rango [-1, 1]) |
| 8 | Colores del overlay mezclaban BGR y RGB | Conversión correcta |
| 9 | Solo funcionaba con recortes de 256×256 | Localización en **imágenes grandes** con ventana deslizante |

> Ejecuta las celdas en orden. Usa un entorno con **GPU** (Entorno de ejecución → Cambiar tipo de entorno → GPU).

In [ ]:
# [SOLO_COLAB]
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, shutil, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, precision_score, recall_score)

# ---------------- Configuración ----------------
RUTA_DRIVE   = Path('/content/drive/MyDrive/IA 2/Dataset_IA2')   # dataset original
RUTA_LOCAL   = Path('/content/dataset')                          # copia local (más rápida que Drive)
RUTA_MODELOS = Path('/content/drive/MyDrive/IA 2/modelos')       # aquí se guardan los modelos
RUTA_MODELOS.mkdir(parents=True, exist_ok=True)

TAM_IMG  = (256, 256)
TAM_LOTE = 32
SEMILLA  = 123
EPOCAS_FASE1 = 5     # solo cabeza de clasificación
EPOCAS_FASE2 = 10    # fine-tuning
LR_FASE1 = 1e-3
LR_FASE2 = 1e-5

random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if not gpus:
    print('⚠️  No hay GPU: el entrenamiento será MUY lento. Activa una GPU en el entorno de ejecución.')

## 1. Organizar el dataset y registrar la foto de origen

Copiamos las imágenes a disco local (si no están ya) y construimos una tabla con la ruta, la etiqueta, la superficie
y un **identificador de grupo** (la foto de la que viene cada recorte). El dataset parece ser SDNET2018, donde cada
imagen de 256×256 es un recorte de una foto más grande y los nombres tienen la forma `7001-115.jpg`
(foto `7001`, recorte `115`). Si tus nombres son distintos, revisa la salida de diagnóstico de la celda.

In [ ]:
# [SOLO_COLAB]
from tqdm.auto import tqdm

SUPERFICIES = {'decks', 'pavements', 'walls'}
EXTENSIONES = {'.jpg', '.jpeg', '.png'}

def id_foto(nombre_archivo):
    """Extrae el id de la foto de origen: '7001-115.jpg' -> '7001'. Si no hay patrón, usa el nombre completo."""
    m = re.search(r'(\d+)[-_]\d+', Path(nombre_archivo).stem)
    return m.group(1) if m else Path(nombre_archivo).stem

registros = []
for carpeta_sup in sorted(RUTA_DRIVE.iterdir()):
    if not (carpeta_sup.is_dir() and carpeta_sup.name.lower() in SUPERFICIES):
        continue
    superficie = carpeta_sup.name
    for carpeta_cls in sorted(carpeta_sup.iterdir()):
        if not carpeta_cls.is_dir():
            continue
        nombre = carpeta_cls.name.lower().replace('-', '_')
        if nombre == 'cracked':
            etiqueta, destino = 1, 'grieta'
        elif nombre == 'non_cracked':
            etiqueta, destino = 0, 'sin_grieta'
        else:
            continue
        carpeta_dest = RUTA_LOCAL / destino
        carpeta_dest.mkdir(parents=True, exist_ok=True)
        archivos = [f for f in carpeta_cls.iterdir() if f.is_file() and f.suffix.lower() in EXTENSIONES]
        for f in tqdm(archivos, desc=f'{superficie}/{carpeta_cls.name}'):
            dest = carpeta_dest / f'{superficie}_{f.name}'
            if not dest.exists():
                shutil.copy(f, dest)
            registros.append({'ruta': str(dest), 'etiqueta': etiqueta, 'superficie': superficie,
                              'grupo': f'{superficie}_{id_foto(f.name)}'})

df = pd.DataFrame(registros)
df.to_csv(RUTA_LOCAL / 'indice.csv', index=False)

In [ ]:
print(f'Total de imágenes: {len(df)}')
print(df.groupby(['superficie', 'etiqueta']).size().unstack(fill_value=0).rename(columns={0: 'sin_grieta', 1: 'grieta'}))
print(f"\n% de imágenes con grieta: {100 * df.etiqueta.mean():.1f} %  "
      f"(un modelo que siempre diga 'sin grieta' ya tendría {100 * (1 - df.etiqueta.mean()):.1f} % de accuracy)")

# Diagnóstico del agrupamiento por foto
print(f"\nGrupos (fotos de origen) detectados: {df.grupo.nunique()}  ->  ~{len(df) / df.grupo.nunique():.1f} recortes por foto")
print('Ejemplos de nombres:', [Path(r).name for r in df.ruta.sample(5, random_state=SEMILLA)])
if df.grupo.nunique() > 0.8 * len(df):
    print('⚠️  Casi cada imagen es su propio grupo: el patrón de nombre no se detectó y el split NO evitará la fuga de datos.')

## 2. División train / validación / test **por foto de origen**

Si recortes de la misma foto caen en train y en validación, el modelo "reconoce" la foto y la métrica sale inflada.
Con `GroupShuffleSplit` todos los recortes de una misma foto quedan en un único conjunto.
El **test** no se toca hasta el final.

In [ ]:
def dividir_por_grupo(datos, fraccion, semilla):
    gss = GroupShuffleSplit(n_splits=1, test_size=fraccion, random_state=semilla)
    idx_a, idx_b = next(gss.split(datos, groups=datos['grupo']))
    return datos.iloc[idx_a].reset_index(drop=True), datos.iloc[idx_b].reset_index(drop=True)

df_trainval, df_test = dividir_por_grupo(df, 0.15, SEMILLA)
df_train, df_val     = dividir_por_grupo(df_trainval, 0.15 / 0.85, SEMILLA)

# Verificación: ningún grupo en dos conjuntos
assert not (set(df_train.grupo) & set(df_val.grupo)), 'Fuga train/val'
assert not (set(df_train.grupo) & set(df_test.grupo)), 'Fuga train/test'
assert not (set(df_val.grupo) & set(df_test.grupo)), 'Fuga val/test'

resumen = pd.DataFrame({
    nombre: {'imágenes': len(d), 'con grieta': int(d.etiqueta.sum()), '% grieta': round(100 * d.etiqueta.mean(), 1),
             'fotos (grupos)': d.grupo.nunique()}
    for nombre, d in [('train', df_train), ('val', df_val), ('test', df_test)]})
resumen

## 3. Pipeline de datos

- Aumento de datos **solo en entrenamiento** (giros, rotación, brillo, contraste). Una grieta sigue siendo grieta al girarla.
- `preprocess_input` de MobileNetV2 (rango [-1, 1]), que es lo que esperan los pesos de ImageNet.
- **Pesos por muestra** para compensar el desbalance: cada grieta pesa más que una imagen sin grieta.
- La mezcla (`shuffle`) se hace sobre las rutas, antes de cargar las imágenes, para no gastar memoria.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1, fill_mode='reflect'),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
], name='aumento_datos')

# Pesos para compensar el desbalance (n_total / (2 * n_clase))
n_total = len(df_train)
n1 = int(df_train.etiqueta.sum()); n0 = n_total - n1
PESO_CLASE = {0: n_total / (2 * n0), 1: n_total / (2 * n1)}
print('Pesos por clase:', {k: round(v, 2) for k, v in PESO_CLASE.items()})

def cargar_imagen(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, TAM_IMG)
    img.set_shape((*TAM_IMG, 3))
    return img, tf.cast(etiqueta, tf.float32)

def crear_dataset(datos, entrenamiento=False):
    rutas = datos['ruta'].astype(str).values
    etiquetas = datos['etiqueta'].values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if entrenamiento:
        ds = ds.shuffle(len(datos), seed=SEMILLA, reshuffle_each_iteration=True)
    ds = ds.map(cargar_imagen, num_parallel_calls=AUTOTUNE)
    if entrenamiento:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    if entrenamiento:
        w0, w1 = PESO_CLASE[0], PESO_CLASE[1]
        ds = ds.map(lambda x, y: (x, y, tf.where(y > 0.5, w1, w0)), num_parallel_calls=AUTOTUNE)
    return ds.batch(TAM_LOTE).prefetch(AUTOTUNE)

ds_train = crear_dataset(df_train, entrenamiento=True)
ds_val   = crear_dataset(df_val)
ds_test  = crear_dataset(df_test)

# Vista rápida de un lote de entrenamiento (con aumento de datos)
x, y, w = next(iter(ds_train))
fig, ejes = plt.subplots(2, 4, figsize=(14, 7))
for i, eje in enumerate(ejes.ravel()):
    eje.imshow((x[i].numpy() + 1) / 2)          # de [-1,1] a [0,1] solo para mostrar
    eje.set_title(f"{'GRIETA' if y[i] > 0.5 else 'sin grieta'}  (peso {w[i]:.2f})")
    eje.axis('off')
plt.tight_layout(); plt.show()

## 4. Modelo: MobileNetV2 + `GlobalAveragePooling2D` + capa densa

`GlobalAveragePooling2D` conserva la relación entre cada canal del mapa de características y la decisión final,
que es lo que permite calcular el mapa de activación de clase. Las capas de la cabeza tienen nombre fijo
(`gap`, `dropout`, `clasificador`) para poder usarlas luego en Grad-CAM.

> La advertencia sobre `input_shape` y los pesos de 224×224 es normal: la red es totalmente convolucional y funciona con 256×256.

In [ ]:
def construir_modelo():
    base = keras.applications.MobileNetV2(input_shape=(*TAM_IMG, 3), include_top=False, weights='imagenet')
    base.trainable = False

    entradas = keras.Input(shape=(*TAM_IMG, 3), name='imagen')
    x = base(entradas, training=False)               # BatchNorm siempre en modo inferencia
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3, name='dropout')(x)
    salidas = layers.Dense(1, activation='sigmoid', name='clasificador')(x)
    return keras.Model(entradas, salidas, name='detector_grietas'), base

def compilar(modelo, lr):
    modelo.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='accuracy'),
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')])

def crear_callbacks(nombre):
    return [
        keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=1, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(str(RUTA_MODELOS / f'{nombre}.keras'), monitor='val_auc',
                                        mode='max', save_best_only=True),
    ]

modelo, base = construir_modelo()
compilar(modelo, LR_FASE1)
modelo.summary()

## 5. Entrenamiento

**Fase 1:** solo se entrena la cabeza (la base está congelada).
**Fase 2 (fine-tuning):** se descongelan los últimos bloques de MobileNetV2 (desde `block_13`) con una tasa de
aprendizaje muy baja. Las capas `BatchNormalization` se mantienen congeladas.
El mejor modelo se elige por **AUC de validación**, no por accuracy.

In [ ]:
print('=== FASE 1: entrenar la cabeza ===')
hist1 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_FASE1,
                   callbacks=crear_callbacks('fase1_cabeza'))

In [ ]:
print('=== FASE 2: fine-tuning de los últimos bloques ===')
base.trainable = True
nombres_capas = [c.name for c in base.layers]
inicio = nombres_capas.index('block_13_expand')      # primera capa del bloque 13
for i, capa in enumerate(base.layers):
    capa.trainable = (i >= inicio) and not isinstance(capa, layers.BatchNormalization)
print(f'Capas descongeladas: {sum(c.trainable for c in base.layers)} de {len(base.layers)}')

compilar(modelo, LR_FASE2)   # hay que recompilar después de cambiar 'trainable'
hist2 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_FASE2,
                   callbacks=crear_callbacks('fase2_finetuning'))

modelo.save(RUTA_MODELOS / 'modelo_grietas_final.keras')

In [ ]:
# Curvas de entrenamiento (ambas fases juntas)
h = {k: hist1.history[k] + hist2.history[k] for k in hist1.history if k in hist2.history}
corte = len(hist1.history['loss'])
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
for eje, (clave, titulo) in zip(ejes, [('loss', 'Pérdida'), ('auc', 'AUC'), ('recall', 'Recall (grietas encontradas)')]):
    eje.plot(h[clave], label='train'); eje.plot(h['val_' + clave], label='validación')
    eje.axvline(corte - 0.5, color='gray', ls='--', label='inicio fine-tuning')
    eje.set_title(titulo); eje.set_xlabel('época'); eje.legend()
plt.tight_layout(); plt.show()

## 6. Evaluación del clasificador

Con clases desbalanceadas la accuracy engaña. Elegimos el **umbral de decisión con el conjunto de validación**
(el que maximiza F1) y lo aplicamos al conjunto de **test**, que el modelo nunca vio.

In [ ]:
p_val  = modelo.predict(ds_val,  verbose=1).ravel()
p_test = modelo.predict(ds_test, verbose=1).ravel()
y_val, y_test = df_val.etiqueta.values, df_test.etiqueta.values

# Umbral que maximiza F1 en validación
prec, rec, umbrales = precision_recall_curve(y_val, p_val)
f1 = 2 * prec * rec / (prec + rec + 1e-8)
UMBRAL_DECISION = float(umbrales[np.argmax(f1[:-1])])
print(f'Umbral elegido en validación: {UMBRAL_DECISION:.3f}  (F1 val = {f1[:-1].max():.3f})\n')

pred_test = (p_test >= UMBRAL_DECISION).astype(int)
print(classification_report(y_test, pred_test, target_names=['sin_grieta', 'grieta'], digits=3))
print(f'AUC-ROC test: {roc_auc_score(y_test, p_test):.3f}   |   Average Precision (PR-AUC): {average_precision_score(y_test, p_test):.3f}')
print(f'Línea base "siempre sin grieta": accuracy = {100 * (1 - y_test.mean()):.1f} %')

fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test), display_labels=['sin_grieta', 'grieta']).plot(ax=ejes[0], colorbar=False)
ejes[0].set_title('Matriz de confusión (test)')
fpr, tpr, _ = roc_curve(y_test, p_test)
ejes[1].plot(fpr, tpr); ejes[1].plot([0, 1], [0, 1], 'k--'); ejes[1].set_title('Curva ROC'); ejes[1].set_xlabel('FPR'); ejes[1].set_ylabel('TPR')
p_, r_, _ = precision_recall_curve(y_test, p_test)
ejes[2].plot(r_, p_); ejes[2].set_title('Curva Precisión-Recall'); ejes[2].set_xlabel('Recall'); ejes[2].set_ylabel('Precisión')
plt.tight_layout(); plt.show()

In [ ]:
# Rendimiento por tipo de superficie (test)
filas = []
for sup, g in df_test.assign(prob=p_test).groupby('superficie'):
    pred = (g.prob >= UMBRAL_DECISION).astype(int)
    filas.append({'superficie': sup, 'imágenes': len(g), '% grieta': round(100 * g.etiqueta.mean(), 1),
                  'recall': round(recall_score(g.etiqueta, pred, zero_division=0), 3),
                  'precisión': round(precision_score(g.etiqueta, pred, zero_division=0), 3),
                  'AUC': round(roc_auc_score(g.etiqueta, g.prob), 3) if g.etiqueta.nunique() == 2 else None})
pd.DataFrame(filas)

## 7. Localización con Grad-CAM (corregido)

**Qué se corrigió:** el gradiente se calcula respecto al **logit de la clase "grieta"** de la cabeza `clasificador`
(no respecto a la salida de la base, y usando el logit en vez de la probabilidad para evitar que la sigmoide saturada
produzca gradientes nulos).

**Resolución:** la última capa convolucional da un mapa de solo 8×8, demasiado grueso para una grieta fina.
Calculamos Grad-CAM también en capas intermedias (16×16 y 32×32) y las fusionamos con una media geométrica
(heurística: el mapa grueso dice *dónde mirar* y el fino *dónde exactamente*).

In [ ]:
def obtener_base(modelo):
    """Devuelve el sub-modelo MobileNetV2 anidado dentro del modelo completo."""
    return next(l for l in modelo.layers if isinstance(l, keras.Model))

_extractores = {}
def _extractor(modelo, nombre_capa):
    clave = (id(modelo), nombre_capa)
    if clave not in _extractores:
        b = obtener_base(modelo)
        _extractores[clave] = keras.Model(b.input, [b.get_layer(nombre_capa).output, b.output])
    return _extractores[clave]

def grad_cam(img_prep, modelo, nombre_capa='out_relu'):
    """
    img_prep: array (1, 256, 256, 3) ya pasado por preprocess_input.
    Devuelve (mapa_calor en [0,1] con la resolución de la capa, probabilidad de grieta).
    """
    extractor = _extractor(modelo, nombre_capa)
    gap, clasif = modelo.get_layer('gap'), modelo.get_layer('clasificador')
    x = tf.convert_to_tensor(img_prep)
    with tf.GradientTape() as cinta:
        cinta.watch(x)                                   # asegura que se registre toda la cadena
        act_capa, mapas_finales = extractor(x, training=False)
        vector = gap(mapas_finales)
        logit = tf.matmul(vector, clasif.kernel) + clasif.bias   # logit de "grieta"
        objetivo = logit[:, 0]
    grads = cinta.gradient(objetivo, act_capa)
    pesos = tf.reduce_mean(grads, axis=(0, 1, 2))               # importancia de cada canal
    cam = tf.nn.relu(tf.reduce_sum(act_capa[0] * pesos, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return cam, float(tf.sigmoid(logit)[0, 0])

# Capas a usar (resolución para entrada de 256×256)
CAPAS_CAM = {'8×8 (out_relu)': 'out_relu',
             '16×16 (block_13_expand_relu)': 'block_13_expand_relu',
             '32×32 (block_6_expand_relu)': 'block_6_expand_relu'}

def redimensionar_cam(cam, tam=TAM_IMG):
    return np.clip(cv2.resize(cam, (tam[1], tam[0]), interpolation=cv2.INTER_CUBIC), 0, 1)

def fusionar_cams(lista_cams):
    """Media geométrica de mapas ya redimensionados a 256×256."""
    f = np.exp(np.mean(np.log(np.stack(lista_cams) + 1e-3), axis=0))
    return (f - f.min()) / (f.max() - f.min() + 1e-8)

def cam_fusionada(img_prep, modelo):
    cams, prob = [], None
    for capa in CAPAS_CAM.values():
        cam, prob = grad_cam(img_prep, modelo, capa)
        cams.append(redimensionar_cam(cam))
    return fusionar_cams(cams), cams, prob

def binarizar(cam, umbral_cam=0.5):
    mascara = (cam >= umbral_cam).astype(np.uint8)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, k)    # quita puntos aislados
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, k)   # une fragmentos cercanos
    return mascara

def obtener_cajas(mascara, area_minima=50):
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [cv2.boundingRect(c) for c in contornos if cv2.contourArea(c) >= area_minima]   # (x, y, w, h)

def superponer(img, cam, alpha=0.4):
    color = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img, 1 - alpha, color, alpha, 0)

# Comprobación de resoluciones reales de cada capa
b = obtener_base(modelo)
for titulo, capa in CAPAS_CAM.items():
    print(f'{capa:25s} -> salida {b.get_layer(capa).output.shape[1:]}')

In [ ]:
def visualizar(ruta, modelo, umbral_cam=0.5):
    img = np.array(keras.utils.load_img(ruta, target_size=TAM_IMG))            # RGB uint8
    prep = preprocess_input(img[None].astype('float32'))
    fusion, cams, prob = cam_fusionada(prep, modelo)
    mascara = binarizar(fusion, umbral_cam)
    cajas = obtener_cajas(mascara)

    con_cajas = img.copy()
    for (x, y, w, h) in cajas:
        cv2.rectangle(con_cajas, (x, y), (x + w, y + h), (255, 0, 0), 2)

    paneles = [(img, 'Original')]
    paneles += [(superponer(img, c), f'Grad-CAM {t.split(" ")[0]}') for t, c in zip(CAPAS_CAM, cams)]
    paneles += [(superponer(img, fusion), 'Fusión (media geométrica)'),
                (con_cajas, f'Máscara ≥ {umbral_cam} + cajas')]
    fig, ejes = plt.subplots(2, 3, figsize=(14, 9.5))
    for eje, (im, titulo) in zip(ejes.ravel(), paneles):
        eje.imshow(im); eje.set_title(titulo); eje.axis('off')
    veredicto = 'GRIETA' if prob >= UMBRAL_DECISION else 'sin grieta'
    fig.suptitle(f'{Path(ruta).name}   |   P(grieta) = {prob:.3f}  ->  {veredicto}', fontsize=13)
    plt.tight_layout(); plt.show()
    return fusion, mascara

# Ejemplos del conjunto de test: aciertos, falsos negativos y falsos positivos
df_ej = df_test.assign(prob=p_test)
grupos_ej = [
    ('ACIERTOS (grietas detectadas con más confianza)',
     df_ej[(df_ej.etiqueta == 1) & (df_ej.prob >= UMBRAL_DECISION)].sort_values('prob', ascending=False).head(3)),
    ('FALSOS NEGATIVOS (grieta que el modelo no vio)',
     df_ej[(df_ej.etiqueta == 1) & (df_ej.prob < UMBRAL_DECISION)].head(2)),
    ('FALSOS POSITIVOS (sin grieta, pero el modelo dijo grieta)',
     df_ej[(df_ej.etiqueta == 0) & (df_ej.prob >= UMBRAL_DECISION)].head(2)),
    ('VERDADEROS NEGATIVOS',
     df_ej[(df_ej.etiqueta == 0) & (df_ej.prob < UMBRAL_DECISION)].head(1)),
]
for titulo, g in grupos_ej:
    print('\n' + '=' * 90 + f'\n{titulo}\n' + '=' * 90)
    for ruta in g.ruta:
        visualizar(ruta, modelo)

## 8. Localizar grietas en **cualquier** imagen (ventana deslizante)

El modelo se entrenó con recortes de 256×256. Si le das una foto grande de una pared, reducirla a 256×256 haría
desaparecer las grietas finas. Por eso recorremos la foto con ventanas de 256×256 (con solapamiento), clasificamos
cada ventana y aplicamos Grad-CAM solo en las que el modelo marca como grieta. Luego se recompone el mapa completo.

Uso: `resultado = localizar_en_imagen_grande('/content/drive/MyDrive/mi_foto.jpg', modelo)`

In [ ]:
def localizar_en_imagen_grande(ruta, modelo, paso=128, umbral_prob=None, umbral_cam=0.5, max_parches_cam=400):
    umbral_prob = UMBRAL_DECISION if umbral_prob is None else umbral_prob
    img = cv2.cvtColor(cv2.imread(str(ruta)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    t = TAM_IMG[0]
    if min(H, W) < t:                                            # imagen más pequeña que una ventana
        esc = t / min(H, W)
        img = cv2.resize(img, None, fx=esc, fy=esc, interpolation=cv2.INTER_CUBIC)
        H, W = img.shape[:2]

    ys = list(range(0, H - t + 1, paso)); xs = list(range(0, W - t + 1, paso))
    if ys[-1] != H - t: ys.append(H - t)
    if xs[-1] != W - t: xs.append(W - t)
    coords = [(y, x) for y in ys for x in xs]
    print(f'Imagen {W}×{H}px  ->  {len(coords)} ventanas de {t}×{t}')

    # 1) Probabilidad de grieta por ventana (por lotes para no agotar memoria)
    probs = []
    for i in range(0, len(coords), 64):
        lote = np.stack([img[y:y + t, x:x + t] for y, x in coords[i:i + 64]]).astype('float32')
        probs.append(modelo.predict(preprocess_input(lote), verbose=0).ravel())
    probs = np.concatenate(probs)

    mapa_prob = np.zeros((H, W), np.float32); cont = np.zeros((H, W), np.float32)
    for (y, x), p in zip(coords, probs):
        mapa_prob[y:y + t, x:x + t] += p; cont[y:y + t, x:x + t] += 1
    mapa_prob /= np.maximum(cont, 1)

    # 2) Grad-CAM fino solo en las ventanas positivas (las más seguras primero)
    candidatas = [i for i in np.argsort(-probs) if probs[i] >= umbral_prob][:max_parches_cam]
    acum = np.zeros((H, W), np.float32)
    for i in candidatas:
        y, x = coords[i]
        prep = preprocess_input(img[y:y + t, x:x + t][None].astype('float32'))
        fusion, _, _ = cam_fusionada(prep, modelo)
        acum[y:y + t, x:x + t] += fusion * probs[i]
    mapa_cam = acum / np.maximum(cont, 1)
    if mapa_cam.max() > 0:
        mapa_cam /= mapa_cam.max()

    mascara = binarizar(mapa_cam, umbral_cam) if candidatas else np.zeros((H, W), np.uint8)
    cajas = obtener_cajas(mascara, area_minima=200)
    con_cajas = img.copy()
    for (x, y, w, h) in cajas:
        cv2.rectangle(con_cajas, (x, y), (x + w, y + h), (255, 0, 0), max(2, H // 300))

    fig, ejes = plt.subplots(1, 3, figsize=(18, 6))
    ejes[0].imshow(img); ejes[0].set_title('Original')
    ejes[1].imshow(superponer(img, mapa_prob)); ejes[1].set_title('Probabilidad de grieta por ventana')
    ejes[2].imshow(con_cajas); ejes[2].set_title(f'Localización: {len(cajas)} zona(s) con grieta' if cajas else 'No se detectaron grietas')
    for e in ejes: e.axis('off')
    plt.tight_layout(); plt.show()
    return {'mapa_prob': mapa_prob, 'mapa_cam': mapa_cam, 'mascara': mascara, 'cajas': cajas}

# Ejemplo (cambia la ruta por una foto tuya):
# resultado = localizar_en_imagen_grande('/content/drive/MyDrive/IA 2/mi_foto_pared.jpg', modelo)

## 9. ¿Qué tan buena es la localización?

Como el entrenamiento no usa máscaras, la calidad de la localización **no se puede medir con el dataset original**.
Para poder reportarla, anota a mano una muestra pequeña (30–50 imágenes con grieta del *test*) con una herramienta
como LabelMe, CVAT o Roboflow, exporta máscaras binarias en PNG (blanco = grieta) y usa la función de abajo.

Dos advertencias para interpretar los números:
- Una grieta es una línea de pocos píxeles y el mapa de calor es una mancha, así que el IoU tenderá a ser bajo
  incluso si la ubicación es correcta. Por eso la función dilata un poco la máscara real (`dilatar_gt`) y también
  reporta qué fracción de la grieta real queda cubierta (`cobertura`).
- Reporta también el caso "sin grieta" (falsos positivos), no solo los aciertos.

In [ ]:
def calcular_metricas_localizacion(mascara_pred, mascara_gt):
    p, g = mascara_pred.astype(bool), mascara_gt.astype(bool)
    inter = np.logical_and(p, g).sum(); union = np.logical_or(p, g).sum()
    return {'IoU': inter / union if union else 1.0,
            'Dice': 2 * inter / (p.sum() + g.sum()) if (p.sum() + g.sum()) else 1.0,
            'cobertura': inter / g.sum() if g.sum() else np.nan}      # % de la grieta real cubierta

def evaluar_localizacion(pares, modelo, umbral_cam=0.5, dilatar_gt=9):
    """pares: lista de (ruta_imagen_256, ruta_mascara_png)."""
    filas = []
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilatar_gt, dilatar_gt))
    for ruta_img, ruta_mask in pares:
        img = np.array(keras.utils.load_img(ruta_img, target_size=TAM_IMG))
        fusion, _, prob = cam_fusionada(preprocess_input(img[None].astype('float32')), modelo)
        pred = binarizar(fusion, umbral_cam)
        gt = cv2.imread(str(ruta_mask), cv2.IMREAD_GRAYSCALE)
        gt = (cv2.resize(gt, (TAM_IMG[1], TAM_IMG[0]), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8)
        gt = cv2.dilate(gt, k)
        filas.append({'imagen': Path(ruta_img).name, 'prob': round(prob, 3), **calcular_metricas_localizacion(pred, gt)})
    res = pd.DataFrame(filas)
    print(res[['IoU', 'Dice', 'cobertura']].mean().round(3).to_string())
    return res

# Ejemplo de uso cuando tengas tus máscaras:
# pares = [('/content/dataset/grieta/Walls_7001-115.jpg', '/content/drive/MyDrive/IA 2/mascaras/Walls_7001-115.png'), ...]
# resultados = evaluar_localizacion(pares, modelo)

## 10. Cargar el modelo guardado más adelante

```python
modelo = keras.models.load_model('/content/drive/MyDrive/IA 2/modelos/modelo_grietas_final.keras')
UMBRAL_DECISION = 0.5   # usa el valor que obtuviste en la sección 6
```

## Resumen para la defensa

- **Supervisión débil:** solo se usan etiquetas por imagen (grieta / sin grieta); nunca se entrena con coordenadas ni máscaras.
- **Localización:** Grad-CAM sobre el logit de "grieta", combinando tres resoluciones, más binarización y cajas.
- **Validación honesta:** split por foto de origen, umbral elegido en validación, métricas adecuadas para clases desbalanceadas y test intacto hasta el final.
- **Limitación:** el mapa de calor es más grueso que la grieta real; mejorarlo requeriría entrenar un modelo de segmentación con las máscaras pseudo-etiquetadas por Grad-CAM (siguiente paso natural).